# 05. 종합 인사이트 & 액션 아이템

**Dataset**: Google Merchandise Store (GA4 Public Dataset)  
**Period**: 2016-08-01 ~ 2017-08-01

## 목표
- 앞선 4개 분석의 핵심 발견을 종합
- Executive Summary 작성
- 개선 기회 우선순위 매트릭스 (Impact vs Effort)
- 데이터 기반 액션 아이템 제안
- 분석 한계 및 추가 분석 방향

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from google.cloud import bigquery

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 120

client = bigquery.Client()
print('BigQuery 연결 성공')

---
## Executive Summary

Google Merchandise Store의 1년간 트래픽 데이터 분석 결과,
다음 5가지 핵심 발견을 도출했습니다.

### Finding 1: 상품 조회 → 장바구니 단계가 최대 이탈 지점

4단계 이커머스 퍼널 분석 결과, **Product View → Add to Cart** 전환에서 가장 큰 이탈이 발생합니다.
이는 상품 상세 페이지의 CTA(Call to Action)와 가격/배송 정보 전달이 구매 결정에 결정적 영향을 미침을 의미합니다.

### Finding 2: Mobile 전환율이 Desktop보다 유의미하게 낮음

Desktop과 Mobile의 전환율 차이는 **카이제곱 검정으로 통계적 유의성이 확인**되었습니다.
단, Cramer's V(효과 크기)가 작을 수 있으므로 **실질적 차이의 크기**도 함께 해석해야 합니다.
아래 "모바일 기회 크기" 섹션에서 구체적 수치를 확인합니다.

### Finding 3: Week 1 리텐션 드롭이 가장 큼

코호트 리텐션 분석 결과, **첫 방문 후 1주 이내에 가장 큰 리텐션 하락**이 발생합니다.
이는 첫 방문 경험(First-Time User Experience)의 개선이 장기 리텐션의 핵심임을 시사합니다.

### Finding 4: 구매 경험이 리텐션을 향상

구매자와 비구매자의 리텐션 차이는 `03_retention.ipynb`에서 정량화했습니다.
구매 경험이 재방문율을 높이는 효과가 확인되며, 첫 구매 유도 인센티브의 ROI를 뒷받침합니다.

### Finding 5: 상위 구매자에 매출이 집중 (파레토 법칙)

구매자 중 상위 10%가 전체 매출의 상당 부분을 차지합니다.
정확한 비율은 아래 "구매자 재방문 가치" 섹션에서 데이터로 확인합니다.

---
## KPI Dashboard

In [ ]:
query_kpi = """
SELECT
  COUNT(*) AS total_sessions,
  COUNT(DISTINCT fullVisitorId) AS unique_visitors,
  ROUND(COUNTIF(totals.transactions > 0) * 100.0 / COUNT(*), 4) AS overall_cvr_pct,
  ROUND(SUM(totals.totalTransactionRevenue) / 1e6, 2) AS total_revenue_usd,
  ROUND(
    SUM(totals.totalTransactionRevenue) / NULLIF(COUNTIF(totals.transactions > 0), 0) / 1e6, 2
  ) AS aov_usd,
  ROUND(AVG(totals.pageviews), 2) AS avg_pageviews,
  ROUND(COUNTIF(totals.bounces = 1) * 100.0 / COUNT(*), 2) AS bounce_rate_pct,
  ROUND(SUM(totals.totalTransactionRevenue) / 1e6 / COUNT(DISTINCT fullVisitorId), 2)
    AS revenue_per_visitor_usd
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
"""

df_kpi = client.query(query_kpi).to_dataframe()
kpi = df_kpi.iloc[0]

print('='*50)
print('       EXECUTIVE KPI DASHBOARD')
print('       2016-08 ~ 2017-08')
print('='*50)
print(f"  Total Sessions:      {kpi['total_sessions']:>12,.0f}")
print(f"  Unique Visitors:     {kpi['unique_visitors']:>12,.0f}")
print(f"  Conversion Rate:     {kpi['overall_cvr_pct']:>11.4f}%")
print(f"  Total Revenue:       ${kpi['total_revenue_usd']:>11,.2f}")
print(f"  AOV:                 ${kpi['aov_usd']:>11,.2f}")
print(f"  Revenue/Visitor:     ${kpi['revenue_per_visitor_usd']:>11,.2f}")
print(f"  Avg Pageviews:       {kpi['avg_pageviews']:>12.2f}")
print(f"  Bounce Rate:         {kpi['bounce_rate_pct']:>11.2f}%")
print('='*50)

---
## 모바일 기회 크기 산출

In [ ]:
query_mobile_opp = """
WITH device_metrics AS (
  SELECT
    device.deviceCategory AS device,
    COUNT(*) AS sessions,
    COUNTIF(totals.transactions > 0) AS purchases,
    SUM(totals.totalTransactionRevenue) / 1e6 AS revenue_usd,
    ROUND(COUNTIF(totals.transactions > 0) * 100.0 / COUNT(*), 4) AS cvr_pct
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY device
)
SELECT
  mobile.sessions AS mobile_sessions,
  mobile.cvr_pct AS mobile_cvr,
  desktop.cvr_pct AS desktop_cvr,
  ROUND(mobile.sessions * (desktop.cvr_pct - mobile.cvr_pct) / 100, 0) AS additional_purchases,
  ROUND(
    mobile.sessions * (desktop.cvr_pct - mobile.cvr_pct) / 100
    * (desktop.revenue_usd / NULLIF(desktop.purchases, 0)), 2
  ) AS additional_revenue_usd
FROM
  (SELECT * FROM device_metrics WHERE device = 'mobile') mobile,
  (SELECT * FROM device_metrics WHERE device = 'desktop') desktop
"""

df_opp = client.query(query_mobile_opp).to_dataframe()
opp = df_opp.iloc[0]

print(f"모바일 세션 수: {opp['mobile_sessions']:,.0f}")
print(f"모바일 전환율: {opp['mobile_cvr']:.4f}%")
print(f"데스크톱 전환율: {opp['desktop_cvr']:.4f}%")
print(f"")
print(f"모바일 전환율 → 데스크톱 수준 달성 시:")
print(f"  추가 구매 건수: {opp['additional_purchases']:,.0f}건")
print(f"  추가 매출: ${opp['additional_revenue_usd']:,.2f}")

---
## 월별 핵심 지표 추이

In [ ]:
query_trend = """
SELECT
  FORMAT_DATE('%Y-%m', PARSE_DATE('%Y%m%d', date)) AS month,
  COUNT(*) AS sessions,
  ROUND(COUNTIF(totals.transactions > 0) * 100.0 / COUNT(*), 4) AS cvr_pct,
  ROUND(SUM(totals.totalTransactionRevenue) / 1e6, 2) AS revenue_usd,
  ROUND(
    SUM(totals.totalTransactionRevenue) / NULLIF(COUNTIF(totals.transactions > 0), 0) / 1e6, 2
  ) AS aov_usd
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY month
ORDER BY month
"""

df_trend = client.query(query_trend).to_dataframe()
df_trend

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sessions
axes[0,0].bar(df_trend['month'], df_trend['sessions'], color='#4e79a7')
axes[0,0].set_title('Monthly Sessions')
axes[0,0].tick_params(axis='x', rotation=45)

# Revenue
axes[0,1].bar(df_trend['month'], df_trend['revenue_usd'], color='#59a14f')
axes[0,1].set_title('Monthly Revenue (USD)')
axes[0,1].tick_params(axis='x', rotation=45)

# CVR
axes[1,0].plot(df_trend['month'], df_trend['cvr_pct'], marker='o', color='#e15759', linewidth=2)
axes[1,0].set_title('Monthly Conversion Rate (%)')
axes[1,0].tick_params(axis='x', rotation=45)

# AOV
axes[1,1].plot(df_trend['month'], df_trend['aov_usd'], marker='s', color='#f28e2b', linewidth=2)
axes[1,1].set_title('Monthly AOV (USD)')
axes[1,1].tick_params(axis='x', rotation=45)

plt.suptitle('Monthly KPI Trends', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 채널 효율성 매트릭스

In [ ]:
query_channel = """
SELECT
  channelGrouping AS channel,
  COUNT(*) AS sessions,
  ROUND(COUNTIF(totals.transactions > 0) * 100.0 / COUNT(*), 4) AS cvr_pct,
  ROUND(SUM(totals.totalTransactionRevenue) / 1e6, 2) AS revenue_usd,
  ROUND(SUM(totals.totalTransactionRevenue) / 1e6 / COUNT(*), 4) AS rps_usd
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY channel
ORDER BY rps_usd DESC
"""

df_ch = client.query(query_channel).to_dataframe()
df_ch['efficiency'] = pd.cut(
    df_ch['rps_usd'],
    bins=[-np.inf, 0.1, 0.5, np.inf],
    labels=['Low', 'Medium', 'High']
)
df_ch

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
color_map = {'High': '#59a14f', 'Medium': '#f28e2b', 'Low': '#e15759'}
colors = [color_map.get(e, '#999999') for e in df_ch['efficiency']]

bars = ax.barh(df_ch['channel'], df_ch['rps_usd'], color=colors)
ax.set_xlabel('Revenue per Session (USD)')
ax.set_title('Channel Efficiency: Revenue per Session', fontsize=14)

for bar, rps in zip(bars, df_ch['rps_usd']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'${rps:.4f}', va='center', fontsize=9)

legend_handles = [mpatches.Patch(color=c, label=l) for l, c in color_map.items()]
ax.legend(handles=legend_handles, title='Efficiency')
plt.tight_layout()
plt.show()

---
## 구매자 재방문 가치

In [ ]:
query_repeat = """
WITH purchaser_visits AS (
  SELECT
    fullVisitorId,
    COUNT(*) AS total_visits,
    COUNTIF(totals.transactions > 0) AS purchase_visits,
    SUM(totals.totalTransactionRevenue) / 1e6 AS total_revenue_usd
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY fullVisitorId
  HAVING COUNTIF(totals.transactions > 0) > 0
)
SELECT
  CASE
    WHEN purchase_visits = 1 THEN '1-time buyer'
    WHEN purchase_visits = 2 THEN '2-time buyer'
    ELSE '3+ time buyer'
  END AS buyer_segment,
  COUNT(*) AS users,
  ROUND(AVG(total_visits), 1) AS avg_total_visits,
  ROUND(AVG(total_revenue_usd), 2) AS avg_revenue,
  ROUND(SUM(total_revenue_usd), 2) AS total_revenue
FROM purchaser_visits
GROUP BY buyer_segment
ORDER BY buyer_segment
"""

df_repeat = client.query(query_repeat).to_dataframe()
df_repeat['revenue_share'] = df_repeat['total_revenue'] / df_repeat['total_revenue'].sum() * 100
df_repeat

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#4e79a7', '#f28e2b', '#59a14f']

# 사용자 수
axes[0].bar(df_repeat['buyer_segment'], df_repeat['users'], color=colors)
axes[0].set_title('Number of Buyers')
axes[0].set_ylabel('Users')

# 매출 기여
axes[1].bar(df_repeat['buyer_segment'], df_repeat['revenue_share'], color=colors)
axes[1].set_title('Revenue Share (%)')
axes[1].set_ylabel('% of Total Revenue')
for i, v in enumerate(df_repeat['revenue_share']):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.suptitle('Buyer Segments: Count vs Revenue Contribution', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Improvement Opportunity Matrix (데이터 기반 동적 생성)
cvr_gap = opp['desktop_cvr'] - opp['mobile_cvr']
addl_rev = opp['additional_revenue_usd']

print("=" * 90)
print("  IMPROVEMENT OPPORTUNITY MATRIX")
print("=" * 90)
print(f"{'Priority':<10} {'Opportunity':<35} {'Expected Impact':<30} {'Source'}")
print("-" * 90)
print(f"{'P0':<10} {'상품 상세 페이지 CTA 개선':<27} {'최대 이탈 지점 해소 → CVR 상승':<24} {'02_funnel'}")
print(f"{'P0':<10} {'모바일 체크아웃 UX 간소화':<27} {'CVR 갭 {cvr_gap:.2f}pp → +${addl_rev:,.0f}':<24} {'02_funnel, 04_seg'}")
print(f"{'P1':<10} {'첫 방문 1주 리타겟팅':<27} {'Week 1 리텐션 개선 → LTV 증대':<24} {'03_retention'}")
print(f"{'P1':<10} {'첫 구매 인센티브':<27} {'구매자 리텐션 우위 활용':<24} {'03_retention'}")

# 리피트 구매자 데이터 기반
if len(df_repeat) > 0:
    repeat_rev_share = df_repeat[df_repeat['buyer_segment'] != '1-time buyer']['revenue_share'].sum()
    print(f"{'P2':<10} {'VIP 로열티 프로그램':<27} {'리피트 구매자 매출 비중 {repeat_rev_share:.1f}%':<24} {'04_segments'}")

# 채널 효율 기반
if len(df_ch) > 0:
    top_ch = df_ch.iloc[0]
    print(f"{'P2':<10} {f'{top_ch[\"channel\"]} 채널 확대':<27} {'RPS ${top_ch[\"rps_usd\"]:.4f} (최고 효율)':<24} {'05_insights'}")

print("=" * 90)
print(f"\n모바일 기회 크기: 전환율을 데스크톱 수준으로 올리면 +{opp['additional_purchases']:,.0f}건, +${addl_rev:,.2f} 추가 매출")

---
## 분석 한계 및 추가 분석 방향

### 한계
1. **UA 스키마**: GA4가 아닌 구 Universal Analytics 스키마로, 이벤트 기반 분석에 제약
2. **광고비 데이터 부재**: ROAS 정확한 계산 불가 (revenue per session으로 대체)
3. **개인정보**: 사용자 레벨 속성(나이, 성별 등) 부재
4. **기간**: 2016-2017 데이터로 현재 트렌드 반영 제한적

### 추가 분석 방향
1. **상품 레벨 분석**: 어떤 상품이 장바구니에서 이탈이 많은지 파악
2. **User Journey Mapping**: 구매자의 전형적인 경로 분석
3. **시간대별 분석**: 시간대에 따른 전환율 변화 (프로모션 타이밍 최적화)
4. **예측 모델**: 구매 확률 예측 (로지스틱 회귀, XGBoost)
5. **A/B 테스트 설계**: 모바일 체크아웃 개선 실험 설계

---
## 이 분석 프로젝트와 FRE의 연결

| 이 프로젝트 | FRE (Funnel & Retention Explorer) |
|:-----------|:---------------------------------|
| BigQuery SQL로 직접 분석 | 동일한 분석을 **자동화된 SaaS 도구**로 제공 |
| 수동 코호트 리텐션 계산 | **원클릭 리텐션 히트맵** 생성 |
| Python으로 통계 검정 | **내장 세그먼트 비교** + AI 인사이트 |
| 일회성 분석 | **반복 가능한 대시보드** + 알림 |

이 SQL 프로젝트가 **"분석 역량"**을 보여준다면,  
FRE는 이 분석을 **"누구나 쉽게 할 수 있도록 제품화"**한 것입니다.